In [0]:
dbutils.secrets.get(scope="financeScope", key="storageAccountKey")

'[REDACTED]'

In [0]:
spark.conf.set("fs.azure.account.key.finanicialstorage2026.dfs.core.windows.net", dbutils.secrets.get(scope="financeScope", key="storageAccountKey"))

In [0]:
dbutils.fs.ls(
  "abfss://bronze@finanicialstorage2026.dfs.core.windows.net/raw_data/"
)

[FileInfo(path='abfss://bronze@finanicialstorage2026.dfs.core.windows.net/raw_data/Sp500.csv', name='Sp500.csv', size=1291379, modificationTime=1778710319000),
 FileInfo(path='abfss://bronze@finanicialstorage2026.dfs.core.windows.net/raw_data/crude-oil-price.csv', name='crude-oil-price.csv', size=22839, modificationTime=1778710342000)]

In [0]:
sp500_df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("abfss://bronze@finanicialstorage2026.dfs.core.windows.net/raw_data/Sp500.csv")

display(sp500_df)

Date,Open,High,Low,Close,Adj Close,Volume
1950-01-03,16.66,16.66,16.66,16.66,16.66,1260000
1950-01-04,16.85,16.85,16.85,16.85,16.85,1890000
1950-01-05,16.93,16.93,16.93,16.93,16.93,2550000
1950-01-06,16.98,16.98,16.98,16.98,16.98,2010000
1950-01-09,17.08,17.08,17.08,17.08,17.08,2520000
1950-01-10,17.030001,17.030001,17.030001,17.030001,17.030001,2160000
1950-01-11,17.09,17.09,17.09,17.09,17.09,2630000
1950-01-12,16.76,16.76,16.76,16.76,16.76,2970000
1950-01-13,16.67,16.67,16.67,16.67,16.67,3330000
1950-01-16,16.719999,16.719999,16.719999,16.719999,16.719999,1460000


In [0]:
sp500_df.show()

+----------+---------+---------+---------+---------+---------+-------+
|      Date|     Open|     High|      Low|    Close|Adj Close| Volume|
+----------+---------+---------+---------+---------+---------+-------+
|1950-01-03|    16.66|    16.66|    16.66|    16.66|    16.66|1260000|
|1950-01-04|    16.85|    16.85|    16.85|    16.85|    16.85|1890000|
|1950-01-05|    16.93|    16.93|    16.93|    16.93|    16.93|2550000|
|1950-01-06|    16.98|    16.98|    16.98|    16.98|    16.98|2010000|
|1950-01-09|    17.08|    17.08|    17.08|    17.08|    17.08|2520000|
|1950-01-10|17.030001|17.030001|17.030001|17.030001|17.030001|2160000|
|1950-01-11|    17.09|    17.09|    17.09|    17.09|    17.09|2630000|
|1950-01-12|    16.76|    16.76|    16.76|    16.76|    16.76|2970000|
|1950-01-13|    16.67|    16.67|    16.67|    16.67|    16.67|3330000|
|1950-01-16|16.719999|16.719999|16.719999|16.719999|16.719999|1460000|
|1950-01-17|16.860001|16.860001|16.860001|16.860001|16.860001|1790000|
|1950-

In [0]:
crude_df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("abfss://bronze@finanicialstorage2026.dfs.core.windows.net/raw_data/crude-oil-price.csv")

display(crude_df)

date,price,percentChange,change
1983-03-01T00:00:00Z,29.27,null,null
1983-04-01T00:00:00Z,30.63,4.646,1.36
1983-05-01T00:00:00Z,30.25,-1.241,-0.38
1983-06-01T00:00:00Z,31.38,3.736,1.13
1983-07-01T00:00:00Z,32.0,1.976,0.62
1983-08-01T00:00:00Z,31.59,-1.281,-0.41
1983-09-01T00:00:00Z,30.36,-3.894,-1.23
1983-10-01T00:00:00Z,30.37,0.033,0.01
1983-11-01T00:00:00Z,29.23,-3.754,-1.14
1983-12-01T00:00:00Z,29.6,1.266,0.37


In [0]:
sp500_df = sp500_df.dropDuplicates() ## removes duplicate rows
sp500_df = sp500_df.dropna() ## removes nulls

crude_df = crude_df.dropDuplicates()
crude_df = crude_df.dropna()

In [0]:
crude_df.printSchema()
sp500_df.printSchema()

root
 |-- date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- percentChange: double (nullable = true)
 |-- change: double (nullable = true)

root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Volume: long (nullable = true)



In [0]:
from pyspark.sql.functions import col

sp500_df = sp500_df.withColumnRenamed("Date", "date")
sp500_df = sp500_df.withColumnRenamed("Adj Close", "Adj Close")
sp500_df = sp500_df.toDF(*[c.lower().replace(" ", "_") for c in sp500_df.columns])

crude_df = crude_df.filter(col("price") > 0 ) ##ensures that negaive values are not possible for the price
sp500_df = sp500_df.filter(col("close") > 0 )
sp500_df = sp500_df.filter(col("open") > 0)
sp500_df = sp500_df.filter(col("high") > 0)
sp500_df = sp500_df.filter(col("low") > 0)

In [0]:
from pyspark.sql.functions import *

crude_df = crude_df.withColumn(
    "date",
    to_date(col("date"))
)

crude_df = crude_df.withColumnRenamed("price", "crude_price")

crude_df = crude_df.withColumnRenamed("change", "crude_price_change")
crude_df = crude_df.withColumnRenamed("percentChange", "crude_percent_change")

In [0]:
crude_df.printSchema()
sp500_df.printSchema()

root
 |-- date: date (nullable = true)
 |-- crude_price: double (nullable = true)
 |-- crude_percent_change: double (nullable = true)
 |-- crude_price_change: double (nullable = true)

root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- adj_close: double (nullable = true)
 |-- volume: long (nullable = true)



In [0]:
crude_df.selectExpr("min(date)", "max(date)").show()
sp500_df.selectExpr("min(date)", "max(date)").show() 

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|1983-04-01|2026-05-01|
+----------+----------+

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|1950-01-03|2018-06-06|
+----------+----------+



In [0]:
start_date = "1983-04-01"
end_date = "2018-06-06"

crude_df = crude_df.filter(
    (col("date") >= start_date) & (col("date") <= end_date)
)

sp500_df = sp500_df.filter(
    (col("date") >= start_date) & (col("date") <= end_date)
)

finance_market = crude_df.join(sp500_df, "date", "inner")

In [0]:
crude_df.write.mode("overwrite").parquet(
  "abfss://silver@finanicialstorage2026.dfs.core.windows.net/crude_oil_clean"
)
sp500_df.write.mode("overwrite").parquet(
  "abfss://silver@finanicialstorage2026.dfs.core.windows.net/sp500_clean"
)

## saves to silver container in storage account

In [0]:
crude_silver = spark.read.parquet(
"abfss://silver@finanicialstorage2026.dfs.core.windows.net/crude_oil_clean"
)

sp500_silver = spark.read.parquet(
"abfss://silver@finanicialstorage2026.dfs.core.windows.net/sp500_clean"
)

In [0]:
finance_market_silver_df = crude_silver.join(
    sp500_silver,
    on="date",
    how="inner"
)

In [0]:
finance_market_silver_df.printSchema()

root
 |-- date: date (nullable = true)
 |-- crude_price: double (nullable = true)
 |-- crude_percent_change: double (nullable = true)
 |-- crude_price_change: double (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- adj_close: double (nullable = true)
 |-- volume: long (nullable = true)



In [0]:
display(finance_market_silver_df)

date,crude_price,crude_percent_change,crude_price_change,open,high,low,close,adj_close,volume
1985-08-01,28.08,3.54,0.96,190.919998,192.169998,190.910004,192.110001,192.110001,121500000
1999-03-01,16.76,36.593,4.49,1238.329956,1238.699951,1221.880005,1236.160034,1236.160034,699500000
1999-04-01,18.66,11.337,1.9,1286.369995,1294.540039,1282.560059,1293.719971,1293.719971,703000000
1990-03-01,20.28,-5.85,-1.26,331.890015,334.399994,331.079987,332.73999,332.73999,157930000
1990-11-01,28.85,-18.11,-6.38,303.98999,307.269989,301.609985,307.019989,307.019989,159270000
1999-02-01,12.27,-3.765,-0.48,1279.640015,1283.75,1271.310059,1273.0,1273.0,799400000
1996-04-01,21.2,-1.258,-0.27,645.5,653.869995,645.5,653.72998,653.72998,392120000
1994-11-01,18.05,-0.77,-0.14,472.26001,472.26001,467.640015,468.420013,468.420013,314940000
1997-12-01,17.64,-7.885,-1.51,955.400024,974.77002,955.400024,974.77002,974.77002,590300000
1986-05-01,14.3,7.196,0.96,235.520004,236.009995,234.210007,235.160004,235.160004,146500000


In [0]:
finance_market_silver_df.describe().show()
finance_market_silver_df.count()

+-------+------------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+
|summary|       crude_price|crude_percent_change|crude_price_change|              open|              high|               low|             close|         adj_close|              volume|
+-------+------------------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+
|  count|               269|                 269|               269|               269|               269|               269|               269|               269|                 269|
|   mean| 43.01959107806693|  1.3113382899628259|0.2464312267657993|1009.1057591895901|1016.1379536022305|1002.7923397843857|1011.0905961301113|1011.0905961301113|1.8174024535315986E9|
| stddev|28.851093906614324|   9.383605054652506| 4.718855941373837|  640.8

269

In [0]:
finance_market_silver_df.groupBy("date").count().filter("count > 1").show()

+----+-----+
|date|count|
+----+-----+
+----+-----+



In [0]:
finance_market_silver_df.write.mode("overwrite").parquet(
  "abfss://silver@finanicialstorage2026.dfs.core.windows.net/finance_market"
)
finance_market = spark.read.parquet()

In [0]:
##start of gold layer
from pyspark.sql.functions import year, avg

yearly_trends = finance_market_silver_df.groupBy(
    year("date").alias("year")
).agg(
    avg("crude_price").alias("avg_crude_price"),
    avg("close").alias("avg_sp500_close")
)

yearly_trends.write.mode("overwrite").parquet(
"abfss://gold@finanicialstorage2026.dfs.core.windows.net/yearly_trends"
)

In [0]:
from pyspark.sql.functions import month

monthly_trends = finance_market_silver_df.groupBy(
    month("date").alias("month")
).agg(
    avg("crude_price").alias("avg_crude_price"),
    avg("close").alias("avg_sp500_close")
)

monthly_trends.write.mode("overwrite").parquet(
"abfss://gold@finanicialstorage2026.dfs.core.windows.net/monthly_trends"
)

In [0]:
market_volatility = finance_market_silver_df.withColumn(
    "daily_volatility",
    col("high") - col("low")
)
market_volatility.write.mode("overwrite").parquet(
"abfss://gold@finanicialstorage2026.dfs.core.windows.net/market_volatility"
)

In [0]:
daily_returns = finance_market_silver_df.withColumn(
    "daily_return",
    ((col("close") - col("open")) / col("open")) * 100
)
daily_returns.write.mode("overwrite").parquet(
"abfss://gold@finanicialstorage2026.dfs.core.windows.net/daily_returns"
)

In [0]:
correlation_df = finance_market_silver_df.select(
    col("crude_price"),
    col("close")
)
correlation_df.write.mode("overwrite").parquet(
"abfss://gold@finanicialstorage2026.dfs.core.windows.net/correlation_data"
)

In [0]:
display(correlation_df)

crude_price,close
28.08,192.110001
16.76,1236.160034
18.66,1293.719971
20.28,332.73999
28.85,307.019989
12.27,1273.0
21.2,653.72998
18.05,468.420013
17.64,974.77002
14.3,235.160004


Databricks visualization. Run in Databricks to view.

In [0]:
display(yearly_trends)

year,avg_crude_price,avg_sp500_close
1990,23.955714285714286,333.488569
2003,29.849999999999998,970.9316608333334
2007,74.23285714285714,1484.7200055714286
2018,66.9425,2722.267517
2015,50.74857142857143,2042.4899552857144
2006,66.32625,1313.881240875
2013,97.06125,1619.338745125
1997,19.926666666666666,887.7533468333332
1988,15.708749999999998,267.8549975
1994,17.52125,462.426257875


Databricks visualization. Run in Databricks to view.

In [0]:
display(monthly_trends)

month,avg_crude_price,avg_sp500_close
12,38.922,1054.8075914800002
6,42.13115384615385,1063.1457766153846
3,43.0688,1042.03558768
5,44.065384615384616,1047.123847423077
9,39.247,1017.2280113499999
4,45.50454545454546,1015.0863730454547
8,46.001538461538466,977.3942419999998
7,43.45919999999999,943.5423936800001
10,42.94166666666666,930.6608378333334
11,43.3284,972.7872040400001


Databricks visualization. Run in Databricks to view.

In [0]:

display(daily_returns)

date,crude_price,crude_percent_change,crude_price_change,open,high,low,close,adj_close,volume,daily_return
1985-08-01,28.08,3.54,0.96,190.919998,192.169998,190.910004,192.110001,192.110001,121500000,0.6232992941891916
1999-03-01,16.76,36.593,4.49,1238.329956,1238.699951,1221.880005,1236.160034,1236.160034,699500000,-0.17522971074762905
1999-04-01,18.66,11.337,1.9,1286.369995,1294.540039,1282.560059,1293.719971,1293.719971,703000000,0.571373401787094
1990-03-01,20.28,-5.85,-1.26,331.890015,334.399994,331.079987,332.73999,332.73999,157930000,0.2561014075702073
1990-11-01,28.85,-18.11,-6.38,303.98999,307.269989,301.609985,307.019989,307.019989,159270000,0.9967430177552993
1999-02-01,12.27,-3.765,-0.48,1279.640015,1283.75,1271.310059,1273.0,1273.0,799400000,-0.5188971056051219
1996-04-01,21.2,-1.258,-0.27,645.5,653.869995,645.5,653.72998,653.72998,392120000,1.2749775367931766
1994-11-01,18.05,-0.77,-0.14,472.26001,472.26001,467.640015,468.420013,468.420013,314940000,-0.8131107692137726
1997-12-01,17.64,-7.885,-1.51,955.400024,974.77002,955.400024,974.77002,974.77002,590300000,2.0274225992692685
1986-05-01,14.3,7.196,0.96,235.520004,236.009995,234.210007,235.160004,235.160004,146500000,-0.15285325827355778


Databricks visualization. Run in Databricks to view.

In [0]:
display(market_volatility)

date,crude_price,crude_percent_change,crude_price_change,open,high,low,close,adj_close,volume,daily_volatility
1985-08-01,28.08,3.54,0.96,190.919998,192.169998,190.910004,192.110001,192.110001,121500000,1.259994000000006
1999-03-01,16.76,36.593,4.49,1238.329956,1238.699951,1221.880005,1236.160034,1236.160034,699500000,16.819946000000073
1999-04-01,18.66,11.337,1.9,1286.369995,1294.540039,1282.560059,1293.719971,1293.719971,703000000,11.979980000000069
1990-03-01,20.28,-5.85,-1.26,331.890015,334.399994,331.079987,332.73999,332.73999,157930000,3.3200069999999755
1990-11-01,28.85,-18.11,-6.38,303.98999,307.269989,301.609985,307.019989,307.019989,159270000,5.660004000000015
1999-02-01,12.27,-3.765,-0.48,1279.640015,1283.75,1271.310059,1273.0,1273.0,799400000,12.43994100000009
1996-04-01,21.2,-1.258,-0.27,645.5,653.869995,645.5,653.72998,653.72998,392120000,8.369995000000017
1994-11-01,18.05,-0.77,-0.14,472.26001,472.26001,467.640015,468.420013,468.420013,314940000,4.619995000000017
1997-12-01,17.64,-7.885,-1.51,955.400024,974.77002,955.400024,974.77002,974.77002,590300000,19.369996000000015
1986-05-01,14.3,7.196,0.96,235.520004,236.009995,234.210007,235.160004,235.160004,146500000,1.7999880000000132


Databricks visualization. Run in Databricks to view.